# Evaluating Search Quality: Hit Rate & MRR

Module 01/02 built keyword and vector search but never measured whether the results were actually good — we just eyeballed a few queries. Now we have 395 ground-truth `(question, document)` pairs from `01-data-gen.ipynb`, so we can turn "does this look right?" into two numbers: **Hit Rate** (did the right document show up at all?) and **MRR** (how high up did it show up?). We then use those numbers to tune the keyword-search boosts objectively instead of by intuition.

## 1. Loading the Ground Truth

`ground_truth-new.csv` holds the 395 `(question, document)` pairs, converted to a list of dicts for easy iteration.

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("./data/ground_truth.csv") #-data.csv")

In [ ]:
df_ground_truth.head()

In [ ]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
ground_truth[10]

## 2. Rebuilding the Search Index

Same `minsearch` index as module 01: `load_faq_data()` + filter to `llm-zoomcamp` + `build_index(documents)`. `text_search` wraps `index.search()` with a fixed boost (`question: 3.0, section: 0.5`) so it can be passed around as a plain `query -> results` function.

In [ ]:
from zoomcamp.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [ ]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

In [ ]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

## 3. Checking Relevance for One Question

Take the first ground-truth question and its labeled `document` id. Run `text_search`, then check which of the 5 returned results has a matching `id`. Encoding that as a list of 0s and 1s — one per result position — is the core idea behind every metric that follows: `[1, 0, 0, 0, 0]` means the correct document came back in position 1.

In [ ]:
q = ground_truth[0]
q

In [ ]:
doc_id = q['document']
doc_id

In [ ]:
results = text_search(q['question'])
results

In [ ]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

In [ ]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

## 4. From One Question to a Relevance Function

`compute_relevance_text(q)` packages the pattern above into a function: given a ground-truth record, run `text_search` and return the 0/1 relevance list. Spot-checking a few questions (index 0, 11, 50) shows all three possible shapes — a hit at position 1, a hit further down, and a miss (`[0, 0, 0, 0, 0]`) — which is exactly why aggregate metrics are needed instead of reading through individual results by hand.

In [ ]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

In [ ]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 1, 0, 0]

In [ ]:
[0, 0, 1, 0, 0]

In [ ]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 0, 0, 0]

In [ ]:
[0, 0, 0, 0, 0]

## 5. Relevance for the Whole Dataset

`compute_relevance_total_text` loops `compute_relevance_text` over all 395 ground-truth questions, wrapped in `tqdm` since it's 395 search calls. The result is a list of 395 relevance lists — the raw material both metrics below are computed from.

In [ ]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance = compute_relevance_total_text(ground_truth)

In [ ]:
relevance[:15]

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

In [ ]:
sample = relevance_total[:15]

In [ ]:
sample

## 6. Generalizing to Any Search Function

`compute_relevance_text` is hardcoded to `text_search`. `compute_relevance(q, search_function)` and `compute_relevance_total(ground_truth, search_function)` take the search function as a parameter instead, so the exact same evaluation code can later score `text_search_v2`, boosted variants, or (in the homework) vector and hybrid search — without rewriting the loop each time.

In [ ]:
14 / 15

In [ ]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
hit_rate(relevance)

## 7. Hit Rate

**Hit Rate** = the fraction of questions where the correct document appears *anywhere* in the top-5 results — it doesn't care about rank, only whether the document was found at all. On the 15-question sample it's 14/15 ≈ 0.933; over the full 395 questions with the `question: 3.0` boost, it comes out to **≈ 0.899**.

In [ ]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [ ]:
mrr(sample)

In [ ]:
mrr(relevance)

## 8. Mean Reciprocal Rank (MRR)

Hit Rate treats a hit at position 1 the same as a hit at position 5, but users notice the difference. **MRR** rewards finding the right document early: for each question, take `1 / rank` of the first hit (0 if there's no hit), then average across all questions. A hit at position 1 scores 1.0, position 2 scores 0.5, position 5 scores 0.2. Over the full dataset, `text_search` scores **≈ 0.769** — lower than the 0.899 Hit Rate, meaning some hits are landing below position 1.

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(ground_truth, text_search)

## 9. Putting It Together: The `evaluate` Function

`evaluate(ground_truth, search_function)` runs `compute_relevance_total` once and reports both `hit_rate` and `mrr` from the same pass — one call gives the full scorecard for any search configuration. This becomes the single entry point for every comparison below.

In [ ]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
evaluate(ground_truth, text_search_v2)

In [ ]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

## 10. Tuning the Question Boost

Lowering the `question` boost from 3.0 to 2.0 (`text_search_v2`) already improves both metrics (hit_rate 0.909, mrr 0.792 vs. 0.899/0.769) — evidence the original boost was too aggressive. Sweeping `question_boost` over `[0.5, 1.0, 3.0, 5.0, 10.0]` (with `search_boost`) confirms it: **1.0** is the best single value (hit_rate 0.924, mrr 0.814), and MRR keeps falling as the boost grows past that. More weight on `question` isn't always better — it can crowd out useful signal from `answer` and `section`.

In [ ]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

## 11. Grid Search Over All Boosts

`search_boosts` exposes all three boosts (`question`, `answer`, `section`) at once. A brute-force grid search over `question ∈ {1, 2, 5}`, `answer ∈ {1, 2, 4, 10}`, `section ∈ {0.1, 0.2, 0.5}` (36 combinations, each a full 395-question evaluation) finds several tied top configurations around **hit_rate ≈ 0.975, mrr ≈ 0.885** — e.g. `question=1.0, answer=2.0, section=0.1`. That's a solid jump from the untuned baseline (0.899 / 0.769), and it came from measuring, not guessing.

In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)